In [ ]:
# from pyspark.sql import SparkContext - stara wersja Sparka, bo Context !

In [3]:
from pyspark.sql import SparkSession ### tutaj używane są DFy

In [4]:
spark = SparkSession.builder.getOrCreate()

In [5]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [6]:
sc = spark.sparkContext

In [7]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [9]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

In [11]:
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [14]:
### Zadanie 2.2 - Statystyki per kategoria ###

# Policz sumę, minimum i maksimum kwoty dla każdej kategorii.


from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

category_summary = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_kategoria"),
        _min("amount").alias("min_kategoria"),
        _max("amount").alias("max_kategoria"))
    .orderBy("category")
)

category_summary.show()

+-----------+--------------+-------------+-------------+
|   category|suma_kategoria|min_kategoria|max_kategoria|
+-----------+--------------+-------------+-------------+
|elektronika|    1520770.69|          9.0|       9999.0|
|    książki|     851382.08|          5.0|      9107.25|
|     odzież|     849877.55|          5.0|      9696.63|
|    żywność|     789514.43|          5.0|      6916.92|
+-----------+--------------+-------------+-------------+



In [23]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne, podanie kolumny będącej wyznacznikiem czasowym
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [25]:
hourly.printSchema() ### tutaj kolumna zawiera inne wewnętrzne elementy, niemożliwe dla Pandas

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [26]:
( ### rozdzielenie tych wartości ^^^
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [20]:
### Zadanie 3.2 Okna 30-minutowe per sklep ###

# Policz transakcje i sumę per sklep w każdym 30-minutowym oknie. Posortuj po oknie, a w ramach okna po sklepie.


from pyspark.sql.functions import window

w30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_sklep"),
    )
    .orderBy("window", "store")
)

w30min.show(truncate=False)

+------------------------------------------+--------+---------+----------+
|window                                    |store   |liczba_tx|suma_sklep|
+------------------------------------------+--------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22  |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58  |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17 |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95 |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03 |
|{2026-04-12 09:00:00, 20

In [27]:
### Zadanie 3.3 — W której godzinie sklep “Kraków” miał najwyższy przychód? ###

# Filtruj najpierw po sklepie, potem zrób okno godzinne, posortuj malejąco po sumie.


from pyspark.sql.functions import desc

hourly_krk = (
    df.filter(df["store"] == "Kraków")
    .groupBy("store", window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_Kraków"),
    )
    .orderBy(desc("suma_Kraków"))
)

hourly_krk.show(truncate=False)

+------+------------------------------------------+-----------+
|store |window                                    |suma_Kraków|
+------+------------------------------------------+-----------+
|Kraków|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.86  |
|Kraków|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|341327.83  |
|Kraków|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|201259.26  |
+------+------------------------------------------+-----------+



In [29]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [28]:
### Zadanie 4.2 — Porównaj tumbling vs sliding ###

# Policz łączną liczbę wierszy wynikowych w obu podejściach. Dlaczego sliding daje więcej wierszy?

tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: Ponieważ sliding iteruje po czasie, ale co pół godziny (nakładanie się), jako że jest to okno przesuwne, którego krok wynosi 30 min.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") rozdziela wszystkie transakcje pomiędzy sklepy, w których zostały dokonane, natomiast groupBy(window(...), "store")
#               grupuje je najpierw na podstawie czasu transakcji, a następnie wewnątrz danego okna - pomiędzy odpowiednie sklepy.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 (okna mają przedziały domknięte z jednej strony i otwarte z drugiej) 

In [52]:
########################
##### Praca domowa #####
########################

In [49]:
# 1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

from pyspark.sql.functions import asc

min_gdk = (
    df.filter(df["store"] == "Gdańsk")
    .groupBy("store", window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("avg_Gdańsk"),
    )
    .orderBy(asc("avg_Gdańsk"))
)

min_gdk.show(1, truncate=False)

+------+------------------------------------------+----------+
|store |window                                    |avg_Gdańsk|
+------+------------------------------------------+----------+
|Gdańsk|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01    |
+------+------------------------------------------+----------+
only showing top 1 row



In [44]:
# 2. Policz ile transakcji per kategoria było w oknie 09:00–09:30.

category_tx = (
    df.filter((df["timestamp"] >= "2026-04-12 09:00:00") & (df["timestamp"] < "2026-04-12 09:30:00"))
    .groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("window")
)

category_tx.show(truncate=False)

+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+



In [48]:
# 3. Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

w15_tx = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("count_tx"),
    )
    .orderBy(desc("count_tx"))
)

w15_tx.show(1, truncate=False)

+------------------------------------------+--------+
|window                                    |count_tx|
+------------------------------------------+--------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234    |
+------------------------------------------+--------+
only showing top 1 row

